<a href="https://colab.research.google.com/github/Harshdevyadav/Agentic-AI/blob/main/Fine_tuning_Lab_task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import userdata
import os
from huggingface_hub import login

# Accessing the secret from the sidebar
hf_token = userdata.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    print("Successfully logged in to Hugging Face!")
else:
    print("HF_TOKEN not found. Add it to the Secrets sidebar first.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully logged in to Hugging Face!


In [7]:
# Installing the newest 2026-standard libraries for SLM fine-tuning
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-3-1b-it"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model and Tokenizer loaded successfully!")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model and Tokenizer loaded successfully!


In [9]:
from datasets import load_dataset

# Loading 500 samples for a fast lab turnaround
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:500]")

def format_instruction(sample):
    # Formatting for a conversational SLM
    return {"text": f"User: {sample['instruction']}\nContext: {sample['context']}\nAssistant: {sample['response']}"}

dataset = dataset.map(format_instruction)
print("Dataset formatted and ready!")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset formatted and ready!


In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# --------------------------------------------------
#
# --------------------------------------------------

model_name = "google/gemma-2b"  # change if needed

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # ✅ IMPORTANT for T4
    device_map="auto"
)

# --------------------------------------------------
#
# --------------------------------------------------

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# --------------------------------------------------
#
# --------------------------------------------------

sft_config = SFTConfig(
    output_dir="./lab_task_gemma",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=50,
    logging_steps=10,
    fp16=True,      # Use fp16 on T4
    bf16=False,     #  Disable bf16
    report_to="none",
)

# --------------------------------------------------
#
# --------------------------------------------------

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,      # make sure dataset exists
    peft_config=peft_config,
    args=sft_config,
    processing_class=tokenizer,
)

# --------------------------------------------------
#
# --------------------------------------------------

trainer.train()

print("SUCCESS! Training completed on T4.")


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss
10,2.270236
20,2.139275
30,1.891793
40,2.010974
50,1.888799


🎉 SUCCESS! Training completed on T4.


In [16]:

instruction = "Explain the importance of fine-tuning small language models."
context = "SLMs are efficient but need domain-specific data to perform well."
prompt = f"User: {instruction}\nContext: {context}\nAssistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Key changes: Added do_sample, temperature, and repetition_penalty
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    repetition_penalty=1.2,
    top_p=0.9,
    eos_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

User: Explain the importance of fine-tuning small language models.
Context: SLMs are efficient but need domain-specific data to perform well.
Assistant: Fine Fine
